# Aplicación 1 — Detección con YOLO sobre las modalidades de fusión

**TFM (replanteo).** Compara el efecto de la fusión en detección entrenando un YOLO por **modalidad de entrada**: VIS, IR, **método anterior** (Torre Top-Hat) y **método óptimo multiescala** (propuesta). Reporta mAP@0.5, mAP@0.5:0.95, P, R, F1, FPS y tamaño. Ejecutar en Colab con GPU.


## 1. Instalación y código de fusión


In [ ]:
!pip -q install ultralytics roboflow pyyaml opencv-python-headless
import ultralytics, torch; ultralytics.checks(); print('CUDA', torch.cuda.is_available())


## 2. Dataset etiquetado (Roboflow → formato YOLOv8)


In [ ]:
import os
from roboflow import Roboflow

# 1) Exporta tu clave antes de abrir el notebook (NO la escribas aqui):
#      Linux/Mac:  export ROBOFLOW_API_KEY="xxxxxxxx"
#      Windows PS: $env:ROBOFLOW_API_KEY = "xxxxxxxx"
#    La obtienes en Roboflow -> Account > Roboflow Keys (Private API Key).
api_key = os.environ.get("ROBOFLOW_API_KEY")
assert api_key, "Falta ROBOFLOW_API_KEY en el entorno (ver Account > Roboflow Keys)."

# 2) Reemplaza WORKSPACE y PROYECTO por los de la URL de tu dataset:
#      app.roboflow.com/<WORKSPACE>/<PROYECTO>/<VERSION>
#    Mas fiable: copialo de Versions > Download Dataset > Show download code.
WORKSPACE = "TU_WS"          # <-- completar
PROYECTO  = "TU_PROYECTO"    # <-- completar
VERSION   = 1                # <-- ajustar a tu version

rf = Roboflow(api_key=api_key)
dataset = (rf.workspace(WORKSPACE)
             .project(PROYECTO)
             .version(VERSION)
             .download("yolov8"))


## 3. Generar las 4 modalidades (conservando etiquetas)


In [ ]:
import os, shutil, cv2, numpy as np, yaml
from pathlib import Path
import sys; sys.path.append('/content')  # o la ruta a tu repo
from src.fusion import TopHatFusion
from src.fusion.optimal_top_hat import OptimalMultiscaleFusion

BASE = Path(dataset.location)   # dataset YOLO/COCO descargado de Roboflow
def load_gray(p): return cv2.imread(str(p),cv2.IMREAD_GRAYSCALE).astype(np.float32)/255.0
def pair_ir_path(vis_path):
    # AJUSTAR a tu convención de nombres de los pares IR
    return Path(str(vis_path).replace('/images/','/images_ir/'))
def save_gray(a,p): Path(p).parent.mkdir(parents=True,exist_ok=True); cv2.imwrite(str(p),(np.clip(a,0,1)*255).astype('uint8'))

# Las 4 modalidades de entrada a comparar (etiquetas idénticas: imágenes registradas)
FUSERS = {
    'VIS':            lambda v,i: v,
    'IR':             lambda v,i: i,
    'Anterior_TopHat':lambda v,i: TopHatFusion('disk',5).fuse(v,i),                 # método anterior
    'Optimo_Multiescala': lambda v,i: OptimalMultiscaleFusion(n=6,base_radius=2.89,m=0.10).fuse(v,i),  # PROPUESTA
}
def build_variant(name, fn, fmt='yolo'):
    root = BASE.parent/f'ds_{name}'
    for split in ['train','valid','test']:
        si=BASE/split/'images'; sl=BASE/split/'labels'
        if not si.exists(): continue
        for img in si.glob('*.*'):
            v=load_gray(img); irp=pair_ir_path(img)
            i=load_gray(irp) if irp.exists() else v
            if i.shape!=v.shape: i=cv2.resize(i,(v.shape[1],v.shape[0]))
            save_gray(fn(v,i), root/split/'images'/img.name)
            lbl=sl/(img.stem+'.txt')
            if lbl.exists(): (root/split/'labels').mkdir(parents=True,exist_ok=True); shutil.copy(lbl, root/split/'labels'/lbl.name)
    with open(BASE/'data.yaml') as f: y=yaml.safe_load(f)
    y['path']=str(root); y['train']='train/images'; y['val']='valid/images'; y['test']='test/images'
    with open(root/'data.yaml','w') as f: yaml.safe_dump(y,f)
    return str(root/'data.yaml')
data_yamls={n:build_variant(n,fn) for n,fn in FUSERS.items()}
data_yamls


## 4. Entrenar un YOLO por modalidad y comparar


In [ ]:
from ultralytics import YOLO; import pandas as pd
EP,IMG,SEED=50,640,0; res={}
for name,dy in data_yamls.items():
    m=YOLO('yolov8n.pt'); m.train(data=dy,epochs=EP,imgsz=IMG,seed=SEED,project='runs_mod',name=name,exist_ok=True,verbose=False)
    r=m.val(data=dy,split='test'); P,R=r.box.mp,r.box.mr
    res[name]={'mAP50':r.box.map50,'mAP50_95':r.box.map,'P':P,'R':R,'F1':2*P*R/(P+R+1e-9),'speed_ms':sum(r.speed.values())}
df=pd.DataFrame(res).T.round(4); df['FPS']=(1000/df['speed_ms']).round(1)
df.to_csv('runs_mod/yolo_por_modalidad.csv'); df


## 5. Lectura
Comparar mAP entre VIS, IR, Anterior y Óptimo Multiescala aísla el efecto de la fusión. Se espera que la fusión supere a VIS/IR por separado; contrastar si la propuesta (mejor en Qabf/SSIM/Nabf) se traduce en mejor detección.
